In [ ]:
import pandas as pd
from scipy.special import erfcinv
import numpy as np

In [ ]:
x = {"apple", "banana", "cherry"}
y = {"google", "microsoft"}

z = x.intersection(y)

In [ ]:
p = 1
if not p:
    print(1)

In [ ]:
F = -1/(2**(1/2)*erfcinv(3/2))


def to_mad_scale(x, m):
    return abs(x - m)


def find_outliers(df):
    outliers_index = []
    for column in df.columns:
        m = df[column].median()
        x = df[column].apply(to_mad_scale, m=m)
        res = F * x.median()
        down = m - res * 3
        up = m + res * 3
        print(up, down)
        
        outliers_index.extend(list(df.loc[(df[column] >= up) | (df[column] <= down)].index))
     
    return set(outliers_index)


In [ ]:
df = pd.read_csv('data/tmp2.csv')

In [ ]:
df = df[['PTA right 1000Hz AC', 'PTA right 2000Hz AC']]

In [ ]:
df = df.dropna()

In [ ]:
df = df.replace('[^0-9]+', np.nan, regex=True)

In [ ]:
df = df.astype(float)

In [ ]:
df.plot(x='PTA right 1000Hz AC', y='PTA right 2000Hz AC', kind='scatter')

In [ ]:
find_outliers(df)

In [ ]:
df

## Расстояние Кука

In [ ]:
import pandas as pd

#create dataframe
df = pd.DataFrame({'X': [10, 20, 30, 40, 50, 60],
                   'Y': [20, 30, 40, 50, 100, 70]})

In [ ]:
df.iloc[:, 0]

In [ ]:
import statsmodels.api as sm

def find_outliers_cooks(df):
    """
    нахождение выбросов на основе расстояния Кука
    """
    # storing dependant values
    y = df.iloc[:, 1]

    # storing independent values
    x = df.iloc[:, 0]

    # add bias
    x = sm.add_constant(x)
    
    # create and train linear regression model
    sm_model = sm.regression.linear_model.OLS(y, x).fit()
    influence = sm_model.get_influence()
    
    influence_list = influence.cooks_distance[0]
    influence_df = pd.DataFrame(influence_list, columns=["influence"], index=df.index)
    
    original_length = len(df)
    
    cooks_df = df.join(influence_df)
    cooks_threshold = 4/original_length
    cooks_outliers = cooks_df[cooks_df["influence"] > cooks_threshold]
    
    return set(cooks_outliers.index)

In [ ]:
df = df.dropna()

In [ ]:
find_outliers_cooks(df)

In [ ]:


# storing dependant values
Y = df['Y']

# storing independent values
X = df['X']


X = sm.add_constant(X)

# fit the model
model = sm.OLS(Y, X)
model.fit()

In [ ]:

sm_model = sm.regression.linear_model.OLS(Y, X).fit()
influence = sm_model.get_influence()
influence_list = influence.cooks_distance[0]
influence_df = pd.DataFrame(influence_list, columns=["influence"], index=df.index)
# influence_df.index = houses.index
# cooks_df = houses.merge(influence_df, left_index=True, right_index=True)

In [ ]:
cooks_df = df.join(influence_df)

In [ ]:
original_length = len(df)

In [ ]:
cooks_threshold = 4/original_length
cooks_outliers = cooks_df[cooks_df["influence"] > cooks_threshold]
print("Removed:", len(cooks_outliers))
print(f"This is {len(cooks_outliers) / original_length * 100}% of our dataset")

In [ ]:
import numpy as np
np.set_printoptions(suppress=True)

# create instance of influence
influence = model.get_influence()

# get Cook's distance for each observation
cooks_distances = influence.cooks_distance

# print Cook's distances
print(cooks_distances)